# Corto 1
Alina Carías y Ariela Mishaan  
**Github:** https://github.com/ArielaMishaanCohen/Corto1.git 

## Task 1

Dimensión de las imágenes: $W = 256, H = 256, C = 4$ (RGB + infrarrojo). Dos opciones de diseño para el primer bloque, asumiendo que ambas buscan extraer 64 canales de salida ($C_{out} = 64$)

- A (AlexNet): capa convolucional con filtros 7x7, stride 2, padding 0. 
- B (VGG): 3 capas convolucionales secuenciales con filtros 3x3, stride 1, padding 1, seguidas de una capa de max pooling de 2x2 (stride 2)

### 1. Calcular dimensión espacial de salida ($W_{out} \times H_{out}$) para el tensor después de opción A y opción B

**Datos iniciales**
- Entrada: $W = 256, H = 256, C_{in} = 4$
- Fórmula:
  
  $$O = \lfloor \frac{W - K + 2P}{S} \rfloor + 1$$

**Opción A (AlexNet)**

- Una sola convolución: $K = 7, S = 2, P = 0$
- La fórmula: 
  
  $$O = \lfloor \frac{256 - 7 + 2(0)}{2} \rfloor + 1$$

  $$O = \lfloor \frac{249}{2} \rfloor + 1$$

  $$O = \lfloor 124.5 \rfloor + 1$$

  $$O = 124 + 1 = \mathbf{125}$$

- **Tensor de salida:** $125 \times 125 \times 64$ ($W_{out} = H_{out} = 125$, se extraen 64 canales de salida)

**Opción B (VGG)**

- Tres convoluciones 3x3 con: $K = 3, S = 1, P = 1$
- Luego MaxPooling 2x2, Stride 2
- Primera convolución: 
  
  $$O = \lfloor \frac{256 - 3 + 2(1)}{1} \rfloor + 1$$

  $$O = \lfloor \frac{256}{1} \rfloor + 1$$

  $$O = \mathbf{256}$$

  La dimensión se mantiene: $256 \times 256$

- Segunda y tercera convolución: tienen la misma configuración, entonces se mantiene la dimensión: $256 \times 256$

- MaxPooling 2x2, stride 2: $K = 2, S = 2, P = 0$:
  
  $$O = \lfloor \frac{256 - 2}{2} \rfloor + 1$$

  $$O = \lfloor \frac{254}{2} \rfloor + 1$$

  $$O = 127 + 1 = \mathbf{128}$$

- **Tensor de salida:** $128 \times 128 \times 64$ ($W_{out} = H_{out} = 128$, se extraen 64 canales de salida)

### 2. Calcular matemáticamente cuántos parámetros entrenables (pesos) requiere la opción A vs las capas convolucionales de la opción B. Basado en teoría del campo receptivo, justificar por qué la industria estandarizó opción B

En una convolución: 

$$Parámetros = K \times K \times C_{in} \times C_{out}$$

**Opción A (AlexNet)**
- Una sola capa: $K = 7, C_{in} = 4, C_{out} = 64$
  
  $$Parámetros_A = 7 \times 7 \times 4 \times 64$$
  $$= 49 \times 4 \times 64$$
  $$= 196 \times 64 = \mathbf{12544}$$

- Se necesitan **12 544** parámetros.

**Opción B (VGG)**
- Tres convoluciones 3x3
- Primera convolución: entran 4 canales, salen 64: 
  
  $$3 \times 3 \times 4 \times 64$$
  $$9 \times 4 \times 64$$
  $$36\times 64 = \mathbf{2304}$$

- Segunda convolución: entran 64 canales, salen 64
  
  $$3 \times 3 \times 64 \times 64$$
  $$9 \times 4096 = \mathbf{36864}$$

- Tercera convolución: es igual a la segunda, o sea **36864**
- **Total:** $2304 + 36864 +36864 = \mathbf{76032}$ parámetros

**Comparación**
- Opción A: 12 544
- Opción B: 76 032

**Campo receptivo**: una sola capa 7x7 tiene campo receptivo 7x7. Pero tres capas consecutivas tienen un campo receptivo efectivo de 7x7 porque la primera capa ve 3x3, segunda 5x5, tercera 7x7: $3+2+2 = 7$. Aunque la opción B tenga más parámetros, ofrece ventajas fundamentales:

1. **Más no-linealidad**: cada convolución va seguida de una ReLu (la opción A tiene una convolución y solo una ReLU, esta en cambio tiene 3, lo que introduce más no-linealidades y le da mayor capacidad de modelar patrones complejos)
2. **Representación jerárquica**: en vez de aprender directamente patrones grandes (7x7), la red aprende: 
   - 3x3 -> bordes simples
   - combinación -> texturas
   - combinación -> estructuras más complejas. 
3. **Mejor eficiencia estadística**: los filtros pequeños capturan patrones locales, son más generalizables y reducen riesgo de sobreajuste comparado con un filtro grande que aprende patrones muy específicos desde el inicio. 

**CONCLUSIÓN:** aunque la opción B tenga más parámetros, la industrai la estandarizó porque: 
1. Mantiene el mismo campo receptivo efectivo
2. Introduce más no-linealidad
3. Construye represetnaciones jerárquicas más robustas
4. Mejora la capacidad de abstracción profunda
5. Empíricamente generaliza mejor en datasets reales. 


## Task 2

**1.  Redacte una justificación técnica (máximo 2 párrafos) dirigida al gerente de TI explicando por qué SIFT/HOG fallarían en un entorno agrícola real frente a variaciones de luz, ángulos y oclusiones, y cómo los "Mapas de Características" de una red profunda solucionan la variabilidad semántica.**

SIFT y Hog son descriptores que extraen características basadas en gradientes y puntos clave dentro de las imágenes, pero tienen una limitación importante: dependen mucho de las condiciones visuales cuando se toma la foto. En un campo de mango real, las hojas cambian de apariencia según la hora del día (por la luz distinta que hay durante el día), el ángulo desde el que se fotografíen, o si están un poco encima de otras hojas. Por esto, SIFT y HOG generan vectores de características muy distintos para la misma enfermedad, lo que confunde al SVM y baja drásticamente la precisión del sistema. 

Las redes profundas resuelven esto porque aprenden "Mapas de características" en múltiples capas: la primera detecta bordes y texturas básicas, y las más profundas combinan esa información para reconocer patrones complejos como machas o decoloración, independientemente de la luz o el ángulo. Esto es la variabilidad semántica: la misma enfermedad se puede ver diferente visualmente, pero la red aprende a reconocerla igual. Un SVM con SIFT/HOG no tiene esa capacidad de abstracción jerárquica, por eso estos fallarían en un entorno agrícola real.

**2. Si tuviera que elegir estrictamente entre usar la arquitectura AlexNet o VGG-16 para extraer características de las hojas, ¿cuál elegiría y por qué? Justifique su respuesta basándote en la modularidad, la no-linealidad (ReLU) y la capacidad de abstracción de capas profundas.**

Elegiría VGG-16. Aunque AlexNet fue la pionera en el uso de redes convolucionales y es más ligera en términos de cómputo, VGG-16 ofrece ventajas importantes para este tipo de problemas. 

- Primero, su modularidad: VGG-16 está construida con bloques uniformes de convoluciones 3x3 apiladas, lo que hace que la arquitectura sea más ordenada y fácil de adaptar en escenarios de Transfer Learning. 
- Segundo, la profundidad y no-linealidad: ambas redes usan ReLU, pero VGG-16 aplica muchas más capas de activación gracias a sus 16 capas con pesos, frente a las 8 de AlexNet. 
- Por ultimo, la abstracción en capas profundas: al tener más capas convolucionales, VGG-16 construye representaciones más ricas. No se limita a detectar bordes, sino que puede identificar texturas y detalles específicos de enfermedades en las hojas, como manchas, necrosis o cambios de color, que AlexNet podría pasar por alto. 

Esto quiere decir que, para un escenario de Transfer Learning con imágenes de hojas de mango, VGG-16 ofrece características pre-entrenadas mucho más completas, lo que compensa el costo computacional adicional.

## Task 3

### Librerias

In [3]:
import kagglehub
import os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score

path = kagglehub.dataset_download("aryashah2k/mango-leaf-disease-dataset")
print("Path to dataset files:", path)
print(os.listdir(path))

ModuleNotFoundError: No module named 'torchvision'

### Pipeline

In [ ]:
# Transformaciones para entrenamiento 
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# Transformaciones para validación
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

dataset_completo = datasets.ImageFolder(root=path, transform=train_transforms)
clases = dataset_completo.classes
num_clases = len(clases)
print(f"Clases encontradas ({num_clases}):", clases)

# Split 80% train, 20% validación
total = len(dataset_completo)
tam_train = int(0.8 * total)
tam_val = total - tam_train

train_dataset, val_dataset = random_split(dataset_completo, [tam_train, tam_val])
val_dataset.dataset.transform = val_transforms

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)

print(f"Imágenes de entrenamiento: {tam_train} | Validación: {tam_val}")

### Adaptación de VGG-16 con Transfer Learning

In [ ]:
modelo = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

# features
for param in modelo.features.parameters():
    param.requires_grad = False

modelo.classifier = nn.Sequential(
    nn.Linear(25088, 4096),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(4096, 1024),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(1024, num_clases) 
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelo = modelo.to(device)
print("Usando dispositivo:", device)
print(modelo.classifier)

### Entrenamiento

In [ ]:
criterio   = nn.CrossEntropyLoss()
optimizador = optim.Adam(modelo.classifier.parameters(), lr=0.0001)

EPOCAS = 10
historial_train = []
historial_val   = []

for epoca in range(EPOCAS):
    # entrenamiento 
    modelo.train()
    perdida_train = 0
    correctos_train = 0

    for imagenes, etiquetas in train_loader:
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)

        optimizador.zero_grad()
        salidas = modelo(imagenes)
        perdida = criterio(salidas, etiquetas)
        perdida.backward()
        optimizador.step()

        perdida_train += perdida.item()
        correctos_train += (salidas.argmax(1) == etiquetas).sum().item()

    acc_train = correctos_train / tam_train

    # validación 
    modelo.eval()
    perdida_val = 0
    correctos_val = 0

    with torch.no_grad():
        for imagenes, etiquetas in val_loader:
            imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
            salidas = modelo(imagenes)
            perdida = criterio(salidas, etiquetas)
            perdida_val += perdida.item()
            correctos_val += (salidas.argmax(1) == etiquetas).sum().item()

    acc_val = correctos_val / tam_val

    historial_train.append(acc_train)
    historial_val.append(acc_val)

    print(f"Época {epoca+1}/{EPOCAS} | Loss Train: {perdida_train:.3f} | Acc Train: {acc_train:.3f} | Acc Val: {acc_val:.3f}")

### Evaluación y matriz de confusión

In [ ]:
modelo.eval()
todas_etiquetas = []
todas_predicciones = []

with torch.no_grad():
    for imagenes, etiquetas in val_loader:
        imagenes = imagenes.to(device)
        salidas = modelo(imagenes)
        predicciones = salidas.argmax(1).cpu().numpy()
        todas_predicciones.extend(predicciones)
        todas_etiquetas.extend(etiquetas.numpy())

accuracy_final = accuracy_score(todas_etiquetas, todas_predicciones)
print(f"Precisión global en validación: {accuracy_final * 100:.2f}%")

In [ ]:
# Matriz de confusión
cm = confusion_matrix(todas_etiquetas, todas_predicciones)
fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clases)
disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title("Matriz de Confusión")
plt.tight_layout()
plt.show()

In [ ]:
# Curva de accuracy
plt.figure(figsize=(8, 4))
plt.plot(historial_train, label="Train Accuracy")
plt.plot(historial_val,   label="Val Accuracy")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.title("Curva de Entrenamiento")
plt.legend()
plt.show()

## Reporte de Resultados

### Precisión Global (Accuracy)
El modelo alcanzó una precisión de **XX%** en el conjunto de validación
(reemplaza XX con el número que te salió).

### Matriz de Confusión
La matriz de confusión muestra qué tan bien el modelo distingue entre
las distintas enfermedades. Las clases con más errores suelen ser las
que visualmente se parecen entre sí.

### Conclusión sobre Overfitting
Si la accuracy de entrenamiento es mucho más alta que la de validación
(diferencia mayor a ~10%), el modelo está haciendo overfitting: memorizó
los datos de entrenamiento pero no generaliza bien. Para combatirlo,
usamos **Dropout** en las capas fully connected (visto en clase), que
"apaga" neuronas aleatoriamente durante el entrenamiento para que la
red no dependa demasiado de ninguna en particular.